<div style="padding:10px; border:solid 4px #8e44ad; border-radius:10px; background-color:#f3e5f5;">
<table width="100%">
  <tr>
    <td width="10%" style="text-align:left; font-size:large;">🧙</td>
    <td style="text-align:center;">
      <span style="font-size:xx-large; font-weight:bold; color:#6c3483;">Le Choixpeau Magique</span><br>
      <span style="font-size:large; color:#6c3483;">Algorithme des k plus proches voisins</span>
    </td>
    <td width="10%" style="text-align:right; font-size:large;">🏰</td>
  </tr>
</table>
<hr style="border:1px solid #8e44ad;">
<table width="100%">
  <tr>
    <td><b>Classe :</b> Première NSI</td>
    <td><b>Chapitre :</b> Algorithme des k-NN</td>
    <td><b>Durée :</b> 2 x 2h</td>
  </tr>
</table>
</div>


<h1><span class="title-number">I</span>Contexte</h1>


_Le Choixpeau Magique est un chapeau enchanté qui attribue chaque nouvel élève de Poudlard à l'une des quatre maisons : **Gryffondor**, **Serdaigle**, **Serpentard** ou **Poufsouffle**._

_Pour décider, le Choixpeau analyze la personnalité de l'élève. Dans ce TP, on va lui apprendre à faire ça grâce à l'informatique !_

Chaque élève est décrit par **4 caractéristiques**, notées de 1 à 10 :

| Caractéristique | Description |
|---|---|
| `courage` | La bravoure, l'audace |
| `sagesse` | L'intelligence, la curiosité |
| `ambition` | La détermination, le goût du pouvoir |
| `loyaute` | La fidélité, l'amitié |

On dispose de données sur **120 anciens élèves** déjà triés par maison.  
L'objectif : créer un programme capable de prédire la maison d'un **nouvel élève inconnu** à partir de ces données.

<h1><span class="title-number">II</span>Principe de l'algorithme</h1>


L'**algorithme des k plus proches voisins** (k-NN) repose sur une idée simple :

> Pour prédire la maison d'un nouvel élève, on cherche les **k élèves connus qui lui ressemblent le plus** (ses k plus proches voisins), puis on attribue la maison **la plus représentée** parmi eux.

Pour mesurer à quel point deux élèves se ressemblent, on calcule la **distance euclidienne** :

```
d = sqrt( (c1-c2)² + (s1-s2)² + (a1-a2)² + (l1-l2)² )
```

où `c`, `s`, `a`, `l` désignent les valeurs de courage, sagesse, ambition et loyauté.

!!! question Question 1
Deux élèves A et B ont les caractéristiques suivantes :

- A : courage=8, sagesse=5, ambition=3, loyaute=7
- B : courage=6, sagesse=7, ambition=5, loyaute=4

Calculer à la main la **distance (au carré)** entre A et B, en utilisant la formule :
```
d² = (c1-c2)² + (s1-s2)² + (a1-a2)² + (l1-l2)²
```
!!!


<h1><span class="title-number">III</span>Chargement des données</h1>


Télécharge le fichier **eleves_poudlard.csv** et place-le dans le même dossier que ce notebook.

Le fichier contient 120 élèves. Chaque ligne ressemble à ceci :

```
prenom,courage,sagesse,ambition,loyaute,maison
Harry,9,6,4,8,Gryffondor
```

Le code ci-dessous lit ce fichier et crée une **liste de dictionnaires**.
Lire le code attentivement, puis répondre aux questions.

In [ ]:
import csv

def charger_eleves(nom_fichier):
    # ============================================================
    # On ouvre le fichier CSV et on lit chaque ligne
    # csv.DictReader transforme chaque ligne en dictionnaire Python
    # dont les clés sont les noms des colonnes du fichier
    # ============================================================
    eleves = []
    with open(nom_fichier, encoding='utf-8') as f:
        lecteur = csv.DictReader(f)
        for ligne in lecteur:
            eleves.append({
                'prenom'  : ligne['prenom'],
                'courage' : int(ligne['courage']),   # on convertit en entier
                'sagesse' : int(ligne['sagesse']),
                'ambition': int(ligne['ambition']),
                'loyaute' : int(ligne['loyaute']),
                'maison'  : ligne['maison']
            })
    return eleves

eleves = charger_eleves('eleves_poudlard.csv')
print(f"{len(eleves)} élèves chargés")
print('Exemple :', eleves[0])


!!! question Question 2
1. Pourquoi utilise-t-on `int()` pour les valeurs de courage, sagesse, ambition et loyauté ?
2. Quel est le type de la variable `eleves` ? Que contient-elle ?
3. Comment accéder au courage du premier élève de la liste ?
!!!


### Visualisation des données

Avant de programmer l'algorithme, observons à quoi ressemble notre base de données.  
Le code ci-dessous trace des **graphiques** représentant les élèves selon leurs attributs.  
Chaque couleur correspond à une maison. Exécuter la cellule pour voir le graphe.


In [ ]:
import matplotlib.pyplot as plt

# ── Couleurs et marqueurs par maison ─────────────────────────────────────────
styles = {
    'Gryffondor' : {'color': '#c0392b', 'marker': '^', 'label': 'Gryffondor (rouge)'},
    'Serdaigle'  : {'color': '#2980b9', 'marker': 'o', 'label': 'Serdaigle (bleu)'},
    'Serpentard' : {'color': '#27ae60', 'marker': 's', 'label': 'Serpentard (vert)'},
    'Poufsouffle': {'color': '#f39c12', 'marker': 'D', 'label': 'Poufsouffle (orange)'},
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Visualisation des élèves de Poudlard', fontsize=14, fontweight='bold')

# ── Graphique 1 : Courage vs Sagesse ─────────────────────────────────────────
ax1 = axes[0]
for maison, st in styles.items():
    xs = [e['courage']  for e in eleves if e['maison'] == maison]
    ys = [e['sagesse']  for e in eleves if e['maison'] == maison]
    ax1.scatter(xs, ys, color=st['color'], marker=st['marker'],
                label=st['label'], s=70, alpha=0.85, edgecolors='white', linewidths=0.5)
ax1.set_xlabel('Courage', fontsize=11)
ax1.set_ylabel('Sagesse', fontsize=11)
ax1.set_title('Courage vs Sagesse')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3, linestyle=':')

# ── Graphique 2 : Ambition vs Loyauté ────────────────────────────────────────
ax2 = axes[1]
for maison, st in styles.items():
    xs = [e['ambition'] for e in eleves if e['maison'] == maison]
    ys = [e['loyaute']  for e in eleves if e['maison'] == maison]
    ax2.scatter(xs, ys, color=st['color'], marker=st['marker'],
                label=st['label'], s=70, alpha=0.85, edgecolors='white', linewidths=0.5)
ax2.set_xlabel('Ambition', fontsize=11)
ax2.set_ylabel('Loyauté', fontsize=11)
ax2.set_title('Ambition vs Loyauté')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3, linestyle=':')

plt.tight_layout()
plt.show()


!!! question Question 3
Observer les deux graphiques et répondre :

1. Sur le graphique **Courage vs Sagesse**, les élèves de chaque maison sont-ils bien séparés ou plutôt mélangés ?
2. Quelle maison se distingue le mieux des autres sur le graphique **Ambition vs Loyauté** ?
!!!


<h1><span class="title-number">IV</span>Calculer une distance</h1>


!!! question Question 4
Compléter la fonction `distance(e1, e2)` ci-dessous.  
Elle prend en paramètre deux dictionnaires d'élèves et doit renvoyer leur distance euclidienne.  
Utiliser `math.sqrt()` pour la racine carrée.
!!!


In [ ]:
import math

def distance(e1, e2):
    # ============================================================
    # A FAIRE : calculer la distance euclidienne entre e1 et e2
    # en utilisant les 4 caractéristiques : courage, sagesse, ambition, loyaute
    # ============================================================
    pass

# ── Tests ────────────────────────────────────────────────────────────────────
# La distance d'un élève avec lui-même doit être 0.0
print('Test 1 (attendu : 0.0)  :', distance(eleves[0], eleves[0]))
# La distance entre deux élèves différents doit être > 0
print('Test 2 (attendu : > 0) :', distance(eleves[0], eleves[1]))


<h1><span class="title-number">V</span>Trouver les k voisins les plus proches</h1>


!!! question Question 5
Compléter la fonction `trouver_voisins(eleves, cible, k)` qui :

1. Calcule la distance entre `cible` et chacun des élèves connus.
2. Trie les résultats par distance croissante.
3. Renvoie la liste des **maisons** des k voisins les plus proches.

`cible` est un dictionnaire ne contenant que les 4 caractéristiques (sans la clé `maison`).

**Rappel :** `.sort()` sur une liste de tuples trie selon le premier élément du tuple.
!!!


In [ ]:
def trouver_voisins(eleves, cible, k):
    # ── Étape 1 : calculer toutes les distances ───────────────────────────────
    distances = []
    for e in eleves:
        d = distance(e, cible)
        # A FAIRE : ajouter le tuple (d, maison de e) dans la liste distances

    # ── Étape 2 : trier par distance croissante ───────────────────────────────
    # A FAIRE : trier la liste distances avec .sort()

    # ── Étape 3 : récupérer les maisons des k premiers ────────────────────────
    maisons = []
    for i in range(k):
        # A FAIRE : ajouter la maison du i-ème voisin dans maisons
        pass

    return maisons

# ── Test ─────────────────────────────────────────────────────────────────────
cible_test = {'courage': 9, 'sagesse': 3, 'ambition': 4, 'loyaute': 7}
print('3 voisins (attendu : plutôt Gryffondor) :', trouver_voisins(eleves, cible_test, 3))


!!! question Question 6
Que contient la variable `distances` après la boucle `for`, avant le `.sort()` ?
Quel est son type ? Donner un exemple de ce qu'elle pourrait contenir.
!!!


<h1><span class="title-number">VI</span>Choisir la maison majoritaire</h1>


!!! question Question 7
Compléter la fonction `maison_majoritaire(maisons)` qui reçoit une liste de maisons (ex: `['Gryffondor', 'Serdaigle', 'Gryffondor']`) et renvoie la maison qui apparaît le plus souvent.

Utiliser un **dictionnaire de comptage** pour compter les occurrences de chaque maison.
!!!


In [ ]:
def maison_majoritaire(maisons):
    # ── Étape 1 : compter les occurrences de chaque maison ────────────────────
    compteurs = {}
    for m in maisons:
        if m in compteurs:
            # A FAIRE : incrémenter le compteur de la maison m
            pass
        else:
            # A FAIRE : initialiser le compteur de la maison m à 1
            pass

    # ── Étape 2 : retourner la maison avec le plus grand compteur ─────────────
    # A FAIRE
    pass

# ── Tests ────────────────────────────────────────────────────────────────────
print(maison_majoritaire(['Gryffondor', 'Serdaigle', 'Gryffondor']))           # -> Gryffondor
print(maison_majoritaire(['Serpentard', 'Serdaigle', 'Serpentard',
                          'Poufsouffle', 'Serpentard']))                        # -> Serpentard


<h1><span class="title-number">VII</span>L'algorithme complet</h1>


La fonction `knn()` assemble toutes les étapes précédentes.  
Elle est déjà écrite : lire le code, puis exécuter la cellule.

In [ ]:
def knn(eleves, cible, k):
    """Prédit la maison d'un élève à partir des k plus proches voisins."""
    maisons_voisins = trouver_voisins(eleves, cible, k)
    return maison_majoritaire(maisons_voisins)

# ── Tests avec des profils connus ─────────────────────────────────────────────
harry   = {'courage': 9, 'sagesse': 6, 'ambition': 4, 'loyaute': 8}  # -> Gryffondor ?
luna    = {'courage': 5, 'sagesse': 10,'ambition': 5, 'loyaute': 6}  # -> Serdaigle ?
drago   = {'courage': 6, 'sagesse': 6, 'ambition': 9, 'loyaute': 3}  # -> Serpentard ?
neville = {'courage': 5, 'sagesse': 5, 'ambition': 3, 'loyaute': 9}  # -> Poufsouffle ?

for nom, profil in [('Harry', harry), ('Luna', luna), ('Drago', drago), ('Neville', neville)]:
    print(f"{nom} (k=5) -> {knn(eleves, profil, 5)}")


!!! question Question 8
Les prédictions correspondent-elles aux maisons attendues dans Harry Potter ?  
Si une prédiction est incorrecte, proposer une explication.
!!!


<h1><span class="title-number">VIII</span>Influence de k</h1>


!!! question Question 9
Tester le profil de Harry avec différentes valeurs de k : 1, 3, 5, 7, 11.  
Observer si la maison prédite change selon k.
!!!


In [ ]:
harry = {'courage': 9, 'sagesse': 6, 'ambition': 4, 'loyaute': 8}

# A FAIRE : tester knn() pour k = 1, 3, 5, 7, 11 et afficher le résultat
for k in [1, 3, 5, 7, 11]:
    print(f"k={k} -> {knn(eleves, harry, k)}")


!!! question Question 10
1. La prédiction change-t-elle selon la valeur de k ?
2. Pourquoi utilise-t-on souvent un **k impair** ?
3. Que se passerait-il si on choisissait k = 120 (nombre total d'élèves) ?
!!!


<h1><span class="title-number">IX</span>Et toi, quelle est ta maison ?</h1>


!!! question Question 11
Note-toi honnêtement de 1 à 10 sur chaque caractéristique dans la cellule ci-dessous, puis exécute-la pour découvrir ta maison selon l'algorithme !
!!!


In [ ]:
# ── Mets tes propres notes (entre 1 et 10) ────────────────────────────────────
moi = {
    'courage' : 7,   # <-- remplace par ta note
    'sagesse' : 6,   # <-- remplace par ta note
    'ambition': 5,   # <-- remplace par ta note
    'loyaute' : 8    # <-- remplace par ta note
}

for k in [3, 5, 7, 11]:
    print(f"k={k} -> {knn(eleves, moi, k)}")


!!! question Question 12
1. Quelle maison t'a été attribuée ?
2. Es-tu d'accord avec le résultat ? Pourquoi ?
3. Essaie de légèrement modifier tes notes : la maison prédite change-t-elle avec k = 3 ? Et avec k = 11 ?
!!!


---
<p style="text-align:center; font-style:italic; color:#888;">Première NSI — Algorithme des k plus proches voisins</p>
